#Ecom Analytics Pipeline Project

**Purpose of this Notebook**
1. Collection of data : Extracting or collecting data from different source (csv files)
2. Explicitly defines schemas
3. Store data into BRONZE PATH

In [0]:
RAW_PATH='/Volumes/workspace/default/eccomerce/'
BRONZE_PATH='/Volumes/workspace/default/eccomerce/bronze/'
SILVER_PATH='/Volumes/workspace/default/eccomerce/silver/'
GOLD_PATH='/Volumes/workspace/default/eccomerce/gold/'

#KEY OF GCP PROJECT
GCP_PROJECT='decisive-cinema-489218-g0'
BQ_QUERY='ecommerce'
TEMP_GCS_BUCKET='ecommerce-databricks-temp'


#secrets codes
GCP_SECRET_SCOPE='gcp-secret'
GCP_SECRET_KEY='gcp-sa-key'

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import lit

txn_schema=StructType([
    StructField('transaction_id', IntegerType(), True),
    StructField('customer_id', IntegerType(), True),
    StructField('product_id', IntegerType(), True),
    StructField('store_id', IntegerType(), True),
    StructField('transaction_date', TimestampType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('total_amount', DoubleType(), True)
])

cust_schema=StructType([
    StructField('cust_id', IntegerType(), True),
    StructField('customer_name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('region', StringType(), True),
    StructField('singup_date', TimestampType(), True)
])

prod_schema=StructType([
    StructField('product_id', IntegerType(), True),
    StructField('product_name', StringType(), True),
    StructField('category', StringType(), True),
    StructField('price', DoubleType(), True),
    StructField('brand', StringType(), True)  
])

store_schema=StructType([
    StructField('store_id', IntegerType(), True),
    StructField('store_name', StringType(), True),
    StructField('location', StringType(), True),
    StructField('manager', StringType(), True),
    StructField('open_date', TimestampType(), True) 
])

promo_schema=StructType([
    StructField('promotion_id', IntegerType(), True),
    StructField('product_id', IntegerType(), True),
    StructField('discount_percent', DoubleType(), True),
    StructField('promo_start_date', TimestampType(), True),
    StructField('promo_end_date', TimestampType(), True),
    StructField('channel', StringType(), True)
])

fb_schema=StructType([
    StructField('feedback_id', IntegerType(), True),
    StructField('customer_id', IntegerType(), True),
    StructField('pproduct_id', IntegerType(), True),
    StructField('rating', IntegerType(), True),
    StructField('review', StringType(), True),
    StructField('review_date', TimestampType(), True)  
])

txn_df=spark.read.option('header', True).schema(txn_schema).csv(RAW_PATH+'transactions.csv')
cust_df=spark.read.option('header', True).schema(cust_schema).csv(RAW_PATH+'customers.csv')
prod_df=spark.read.option('header', True).schema(prod_schema).csv(RAW_PATH+'products.csv')
store_df=spark.read.option('header', True).schema(store_schema).csv(RAW_PATH+'stores.csv')
promo_df=spark.read.option('header', True).schema(promo_schema).csv(RAW_PATH+'promotions.csv')
fb_df=spark.read.option('header', True).schema(fb_schema).csv(RAW_PATH+'feedbacks.csv')


print('Trnsaction', txn_df.count())
txn_df.printSchema()

Trnsaction 20
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)



In [0]:
txn_df.selectExpr('count(distinct transaction_id) as unique_txn').display()

unique_txn
20


In [0]:
txn_df.filter('transaction_date IS NULL or total_amount IS NULL').display()

transaction_id,customer_id,product_id,store_id,transaction_date,quantity,total_amount


In [0]:
(txn_df.write.mode('overwrite').format('delta').option('overwriteSchema','true').save(BRONZE_PATH+'transactions'))
(cust_df.write.mode('overwrite').format('delta').option('overwriteSchema','true').save(BRONZE_PATH+'customers'))
(prod_df.write.mode('overwrite').format('delta').option('overwriteSchema','true').save(BRONZE_PATH+'products'))
(store_df.write.mode('overwrite').format('delta').option('overwriteSchema','true').save(BRONZE_PATH+'stores'))
(promo_df.write.mode('overwrite').format('delta').option('overwriteSchema','true').save(BRONZE_PATH+'promotions'))
(fb_df.write.mode('overwrite').format('delta').option('overwriteSchema','true').save(BRONZE_PATH+'feedbacks'))